In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix, average_precision_score

best_tree = DecisionTreeClassifier(
    max_depth=optimal_depth,
    min_samples_leaf=optimal_leaf,
    min_samples_split=max(10, optimal_leaf * 2),
    max_features='sqrt',
    class_weight='balanced',
    random_state=42
)
best_tree.fit(X_tr_s, y_train)

y_pred  = best_tree.predict(X_v_s)
y_proba = best_tree.predict_proba(X_v_s)[:, 1]

print("=== REGULARISED TREE ===")
print(classification_report(y_val, y_pred, digits=4))
print(f"PR-AUC : {average_precision_score(y_val, y_proba):.4f}")
tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()
print(f"TP={tp} | FP={fp} | FN={fn} | TN={tn}")
print(f"Catch rate: {tp/(tp+fn):.4f}")

In [ ]:
from sklearn.tree import export_text, plot_tree
import matplotlib.pyplot as plt

tree_rules = export_text(
    best_tree,
    feature_names=[f'feat_{i}' for i in range(X_train.shape[1])],
    max_depth=3
)
print(tree_rules)

plt.figure(figsize=(20, 8))
plot_tree(
    best_tree,
    feature_names=[f'feat_{i}' for i in range(X_train.shape[1])],
    class_names=['legit', 'fraud'],
    filled=True,
    max_depth=3,
    fontsize=9
)
plt.title('Decision Tree — Top 3 Levels')
plt.tight_layout()
plt.show()

# What to look for:
# Does the root split make domain sense?
# Are fraud-dominated leaves being isolated cleanly?
# Does any split fire on an irrelevant feature? = possible leakage

In [ ]:
import pandas as pd

feat_names = [f'feat_{i}' for i in range(X_train.shape[1])]
importances = pd.Series(
    best_tree.feature_importances_, index=feat_names)
print("Top 10 feature importances:")
print(importances.sort_values(ascending=False).head(10).round(4))

# Red flags:
# Single feature with >0.5 importance = possible leakage
# Feature you know is irrelevant appears in top 5 = investigate
# All importances roughly equal = model using noise features

# Permutation importance — more honest
from sklearn.inspection import permutation_importance

perm_imp = permutation_importance(
    best_tree, X_v_s, y_val,
    n_repeats=10,
    scoring='average_precision',
    random_state=42,
    n_jobs=-1
)
perm_series = pd.Series(perm_imp.importances_mean, index=feat_names)
print("\nTop 10 permutation importances:")
print(perm_series.sort_values(ascending=False).head(10).round(4))